# CS781 Phase 3 — Knowledge Distillation from an out-of-fold CRF ensemble

## The two design decisions that make this honest

**1. Out-of-fold teacher targets.** Your frozen run reached train accuracy 0.808 — the teacher
has largely *memorised* the training set. Distilling its in-sample predictions would teach the
student to imitate memorisation. So the teacher is a 5-fold CV ensemble trained inside the 80
training documents, and document *d*'s soft target comes only from the fold model that never
saw *d*. Those targets carry generalisation signal, not recall.

**2. Distil CRF posterior marginals, not emission softmaxes.** `P(y_i | entire document)` from
forward-backward already integrates the transition structure, so it is a *sequence-aware* soft
target rather than a per-sentence belief. Set `kd_mode="emissions"` for the standard
temperature-softmax formulation as an ablation — CELL 11 measures both.


Everything runs on cached embeddings (frozen backbone), so the whole notebook is ~25 min.

In [ ]:
# CELL 1 — ASSETS

from pathlib import Path
from google.colab import drive
from getpass import getpass
from huggingface_hub import HfApi, hf_hub_download
import os, re, sys, subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")

drive_files = []
for root, dirs, files in os.walk(DRIVE_ROOT):
    dirs[:] = [d for d in dirs if d not in {".Trash", ".ipynb_checkpoints", "node_modules"}]
    for fn in files: drive_files.append(Path(root) / fn)

def normalized_name(p): return re.sub(r" \(\d+\)(?=\.)", "", p.name).lower()

def locate_drive_file(filename, preferred_folder=None):
    tgt = filename.lower()
    m = [p for p in drive_files if p.name.lower() == tgt or normalized_name(p) == tgt]
    if preferred_folder:
        pref = [p for p in m if preferred_folder.lower() in str(p.parent).lower()]
        if pref: m = pref
    if not m: raise FileNotFoundError(f"Could not find {filename}")
    m.sort(key=lambda p: (p.name.lower() != tgt, len(p.parts), str(p).lower()))
    return m[0]

CKPT_PATH      = locate_drive_file("best.pt")
TOKENIZER_JSON = locate_drive_file("bpe_40k.json")
TOKENIZER_PY   = locate_drive_file("a1_tokenizer.py")

HF_REPO = "Exploration-Lab/CS-781-Capstone"
HF_TOKEN = getpass("HF read token: ")
api = HfApi()
phase3 = [f for f in api.list_repo_files(repo_id=HF_REPO, repo_type="dataset",
                                          token=HF_TOKEN) if "phase_3_rr/" in f.lower()]

def dl(filename):
    tgt = filename.lower()
    m = [f for f in phase3 if Path(f).name.lower() == tgt] or \
        [f for f in phase3 if normalized_name(Path(f)).lower() == tgt]
    if "test" in tgt:          pref = [f for f in m if "/test/" in f.lower()]
    elif "label_vocab" in tgt: pref = m
    else:                      pref = [f for f in m if "/train/" in f.lower()]
    return Path(hf_hub_download(repo_id=HF_REPO, filename=(pref or m)[0],
                                repo_type="dataset", token=HF_TOKEN))

CL_TRAIN, CL_DEV = dl("CL_train.json"), dl("CL_dev.json")
IT_TRAIN, IT_DEV = dl("IT_train.json"), dl("IT_dev.json")
CL_TEST, IT_TEST = dl("CL_test.json"), dl("IT_test.json")
LABEL_VOCAB = dl("label_vocab.json")
if str(TOKENIZER_PY.parent) not in sys.path: sys.path.insert(0, str(TOKENIZER_PY.parent))
print(f"Decoder: {CKPT_PATH}")

Mounted at /content/drive
HF read token: ··········


CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

Decoder: /content/drive/MyDrive/best.pt


In [ ]:
# CELL 2 — IMPORTS + CONFIG

import json, math, random, zipfile, time, html, unicodedata
from dataclasses import dataclass
from collections import Counter, deque
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.amp import autocast
from sklearn.metrics import f1_score, classification_report
from a1_tokenizer import BPETokenizer

OUTPUT_DIR = DRIVE_ROOT / "CS781_Phase3_KD_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} ({p.total_memory/1024**3:.1f} GiB)")

@dataclass
class Cfg:
    vocab_size: int = 40_000
    d_model: int = 768
    n_layers: int = 19
    n_query_heads: int = 12
    n_kv_heads: int = 4
    d_ff: int = 2048
    max_seq_len: int = 512
    rope_theta: float = 10_000.0
    dropout: float = 0.0

    sent_max_tokens: int = 128
    sent_sub_batch: int = 32
    n_pos_feats: int = 4

    num_labels: int = 13
    bilstm_hidden: int = 256
    bilstm_layers: int = 2
    bilstm_dropout: float = 0.3
    cls_dropout: float = 0.4
    aux_ce_weight: float = 0.5
    class_weight_cap: float = 4.0

    head_lr: float = 1e-3
    crf_lr: float = 5e-3
    weight_decay: float = 0.01
    max_grad_norm: float = 5.0
    swa_window: int = 5

    # teacher
    n_folds: int = 5
    teacher_epochs: int = 50        # fixed, no early stopping -> no selection leakage
    # student / baseline
    student_epochs: int = 60
    patience: int = 10
    seeds: tuple = (42, 1337, 2024)

    # distillation
    kd_mode: str = "marginals"      # "marginals" (CRF posterior) | "emissions" (classic)
    kd_alpha: float = 0.5           # weight on the HARD loss; (1-alpha) on the KD term
    kd_temp: float = 2.0            # only used by kd_mode="emissions"
    DEV_NOISE: float = 0.026        # measured: the dev split alone moves F1 this much

CFG = Cfg()
print(f"KD: mode={CFG.kd_mode}, alpha={CFG.kd_alpha}, temp={CFG.kd_temp}")
print(f"teacher: {CFG.n_folds} folds x {CFG.teacher_epochs}ep | "
      f"student/baseline: {len(CFG.seeds)} seeds x {CFG.student_epochs}ep")

In [ ]:
# CELL 3 — TOKENIZER + clean_text + DATA

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s); torch.backends.cudnn.benchmark = True
seed_everything(42)

tokenizer = BPETokenizer.load(str(TOKENIZER_JSON))
assert tokenizer.vocab_size == CFG.vocab_size
PAD_ID = int(tokenizer.token_to_id["<pad>"])
BOS_ID = int(tokenizer.token_to_id["<s>"])
EOS_ID = int(tokenizer.token_to_id["</s>"])

TAG_RE=re.compile(r"</?[A-Za-z][^<>]{0,200}>"); URL_RE=re.compile(r"https?://\S+|www\.\S+", re.I)
ENT_RE=re.compile(r"&(?:nbsp|amp|lt|gt|quot|apos|#\d+|#x[0-9a-fA-F]+);", re.I)
CTRL_RE=re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")
WSP_RE=re.compile(r"[\u00a0\u1680\u180e\u2000-\u200a\u202f\u205f\u3000]")
ZW_RE=re.compile(r"[\u200b\u200c\u200d\u2060]"); SPACE_RE=re.compile(r"\s+")

def clean_text(t):
    if t is None: return ""
    t = str(t)
    if t.count("\ufffd") > 50: return ""
    t = TAG_RE.sub(" ", t); t = html.unescape(t)
    t = URL_RE.sub(" ", t); t = ENT_RE.sub(" ", t)
    t = t.replace("\ufeff"," ").replace("\ufffd"," ")
    t = CTRL_RE.sub(" ", t); t = WSP_RE.sub(" ", t)
    t = unicodedata.normalize("NFKC", t); t = ZW_RE.sub("", t)
    return SPACE_RE.sub(" ", t).strip()

def encode_sentence(s):
    ids = tokenizer.encode(clean_text(s)) or [PAD_ID]
    return [BOS_ID] + ids[: CFG.sent_max_tokens - 2] + [EOS_ID]

def pos_features(n):
    i = np.arange(n, dtype=np.float32); rel = i / max(n-1, 1)
    return np.stack([rel, rel**2, np.exp(-i/10.0), np.exp(-(n-1-i)/10.0)], 1).astype(np.float32)

with open(LABEL_VOCAB) as f: LABEL2ID = json.load(f)
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID); CFG.num_labels = NUM_LABELS

def load_lab(p):
    return [{"doc_id": d["id"], "sentences": d["text"],
             "labels": [LABEL2ID[l] for l in d["labels"]]} for d in json.load(open(p))]
def load_test(p):
    return [{"doc_id": k, "sentences": v, "labels": None} for k, v in json.load(open(p)).items()]

train_docs = load_lab(CL_TRAIN) + load_lab(IT_TRAIN)
for d in load_lab(CL_TRAIN): d["domain"] = "CL"
cl_dev, it_dev = load_lab(CL_DEV), load_lab(IT_DEV)
n_cl_train = len(load_lab(CL_TRAIN))
for i, d in enumerate(train_docs): d["domain"] = "CL" if i < n_cl_train else "IT"
dev_docs = cl_dev + it_dev
test_docs = load_test(CL_TEST) + load_test(IT_TEST)

for grp in (train_docs, dev_docs, test_docs):
    for d in grp:
        d["toks"] = [encode_sentence(s) for s in d["sentences"]]
        d["pos"]  = pos_features(len(d["sentences"]))
print(f"train {len(train_docs)}d/{sum(len(d['toks']) for d in train_docs)}s  "
      f"dev {len(dev_docs)}d  test {len(test_docs)}d")

cnt = Counter(l for d in train_docs for l in d["labels"])
f_max = max(cnt.values())
class_weights = torch.tensor(
    [min(CFG.class_weight_cap, math.sqrt(f_max/max(cnt.get(i,1),1)))
     for i in range(NUM_LABELS)], dtype=torch.float32).to(DEVICE)
D_IN = 2*CFG.d_model + CFG.n_pos_feats
ytd = [l for d in dev_docs for l in d["labels"]]
macro_f1 = lambda a, b: f1_score(a, b, labels=list(range(NUM_LABELS)),
                                  average="macro", zero_division=0)

train 80d/16845s  dev 10d  test 10d


In [ ]:
# CELL 4 — DECODER (Phase 1 exact) + FREEZE + CACHE EMBEDDINGS ONCE

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.eps=eps; self.weight=nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float()*rms).to(x.dtype) * self.weight

def precompute_rope(dim, max_len, theta=10000.0):
    freqs = 1.0/(theta ** (torch.arange(0, dim, 2).float()/dim))
    a = torch.outer(torch.arange(max_len).float(), freqs)
    return a.cos(), a.sin()

def apply_rope(x, cos, sin, offset=0):
    T = x.size(2)
    c = cos[offset:offset+T].unsqueeze(0).unsqueeze(0)
    s = sin[offset:offset+T].unsqueeze(0).unsqueeze(0)
    hd = x.size(-1)//2
    x1, x2 = x[..., :hd], x[..., hd:]
    rot = torch.cat([-x2, x1], -1)
    ce = torch.cat([c.expand(-1,-1,-1,hd), c.expand(-1,-1,-1,hd)], -1)
    se = torch.cat([s.expand(-1,-1,-1,hd), s.expand(-1,-1,-1,hd)], -1)
    return x*ce + rot*se

class GQA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nqh=cfg.n_query_heads; self.nkh=cfg.n_kv_heads
        self.ng=cfg.n_query_heads//cfg.n_kv_heads
        self.hd=cfg.d_model//cfg.n_query_heads; self.dm=cfg.d_model
        self.q=nn.Linear(cfg.d_model, self.nqh*self.hd, bias=False)
        self.k=nn.Linear(cfg.d_model, self.nkh*self.hd, bias=False)
        self.v=nn.Linear(cfg.d_model, self.nkh*self.hd, bias=False)
        self.o=nn.Linear(self.nqh*self.hd, cfg.d_model, bias=False)
        self.attn_drop=nn.Dropout(cfg.dropout); self.resid_drop=nn.Dropout(cfg.dropout)
        self.scale=1.0/math.sqrt(self.hd)
    def forward(self, x, cos, sin, mask):
        B,T,_ = x.shape
        q=self.q(x).view(B,T,self.nqh,self.hd).transpose(1,2)
        k=self.k(x).view(B,T,self.nkh,self.hd).transpose(1,2)
        v=self.v(x).view(B,T,self.nkh,self.hd).transpose(1,2)
        q=apply_rope(q,cos,sin); k=apply_rope(k,cos,sin)
        if self.ng > 1:
            k=k.unsqueeze(2).expand(B,self.nkh,self.ng,T,self.hd).reshape(B,self.nqh,T,self.hd)
            v=v.unsqueeze(2).expand(B,self.nkh,self.ng,T,self.hd).reshape(B,self.nqh,T,self.hd)
        w = torch.matmul(q, k.transpose(-2,-1))*self.scale
        if mask is not None: w = w + mask
        w = F.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)
        out = torch.matmul(w, v).transpose(1,2).contiguous().view(B,T,self.dm)
        return self.resid_drop(self.o(out))

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate=nn.Linear(cfg.d_model,cfg.d_ff,bias=False)
        self.up  =nn.Linear(cfg.d_model,cfg.d_ff,bias=False)
        self.down=nn.Linear(cfg.d_ff,cfg.d_model,bias=False)
        self.drop=nn.Dropout(cfg.dropout)
    def forward(self, x): return self.drop(self.down(F.silu(self.gate(x))*self.up(x)))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model); self.attn=GQA(cfg)
        self.norm2=RMSNorm(cfg.d_model); self.ffn=SwiGLU(cfg)
    def forward(self, x, cos, sin, mask):
        x = x + self.attn(self.norm1(x), cos, sin, mask)
        return x + self.ffn(self.norm2(x))

class LegalDecoderBackbone(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb=nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop=nn.Dropout(cfg.dropout)
        self.layers=nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm=RMSNorm(cfg.d_model)
        cos,sin = precompute_rope(cfg.d_model//cfg.n_query_heads, cfg.max_seq_len*2, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
    def forward(self, input_ids, padding_mask=None):
        B,T = input_ids.shape
        x = self.drop(self.tok_emb(input_ids))
        causal = torch.triu(torch.ones(T,T,device=input_ids.device,dtype=torch.bool), diagonal=1)
        mask = torch.zeros((T,T),device=input_ids.device,
                           dtype=x.dtype).masked_fill(causal, float("-inf"))
        mask = mask[None,None,:,:].expand(B,1,T,T).clone()
        if padding_mask is not None:
            mask = mask.masked_fill(padding_mask[:,None,None,:], float("-inf"))
        for layer in self.layers: x = layer(x, self.rope_cos, self.rope_sin, mask)
        return self.norm(x)

decoder = LegalDecoderBackbone(CFG).to(DEVICE)
ck = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
st = ck["model"] if "model" in ck else ck
r = decoder.load_state_dict({k:v for k,v in st.items() if not k.startswith("lm_head")}, strict=False)
assert len(r.missing_keys) == 0, r.missing_keys
print(f"Checkpoint step={ck.get('step','?')} val_loss={ck.get('val_loss','?')}")
for p in decoder.parameters(): p.requires_grad = False
decoder.eval()

@torch.no_grad()
def encode_toks(tok_lists):
    n = len(tok_lists)
    order = sorted(range(n), key=lambda i: len(tok_lists[i]))
    out = np.zeros((n, 2*CFG.d_model), dtype=np.float32)
    for s in range(0, n, CFG.sent_sub_batch):
        idxs = order[s:s+CFG.sent_sub_batch]
        chunk = [tok_lists[i] for i in idxs]
        L = max(len(t) for t in chunk)
        ids = torch.full((len(chunk),L), PAD_ID, dtype=torch.long, device=DEVICE)
        pm  = torch.ones((len(chunk),L), dtype=torch.bool, device=DEVICE)
        last = torch.zeros(len(chunk), dtype=torch.long, device=DEVICE)
        for j,t in enumerate(chunk):
            ids[j,:len(t)] = torch.tensor(t, dtype=torch.long, device=DEVICE)
            pm[j,:len(t)] = False; last[j] = len(t)-1
        with autocast(device_type="cuda", dtype=torch.float16):
            h = decoder(ids, padding_mask=pm)
        h = h.float()
        lv = h[torch.arange(len(chunk), device=DEVICE), last]
        mf = (~pm).unsqueeze(-1).float()
        mv = (h*mf).sum(1)/mf.sum(1).clamp(min=1)
        out[idxs] = torch.cat([lv, mv], -1).cpu().numpy()
    return out

t0 = time.time()
for grp in (train_docs, dev_docs, test_docs):
    for d in grp:
        d["x"] = torch.from_numpy(np.concatenate([encode_toks(d["toks"]), d["pos"]], 1))
print(f"Cached embeddings in {(time.time()-t0)/60:.1f} min | dim {D_IN}")
del decoder; torch.cuda.empty_cache()
print("Backbone freed — everything from here trains in seconds.")

Checkpoint step=40000 val_loss=2.2810269522666933
Cached embeddings in 0.4 min | dim 1540
Backbone freed — everything from here trains in seconds.


In [ ]:
# CELL 5 — CRF with FORWARD-BACKWARD (posterior marginals) + HEAD

class CRF(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.num_tags = C
        self.trans = nn.Parameter(torch.zeros(C, C))
        self.start = nn.Parameter(torch.zeros(C))
        self.end   = nn.Parameter(torch.zeros(C))

    def _gold_score(self, emis, tags):
        T = tags.size(0)
        s = self.start[tags[0]] + self.end[tags[-1]]
        s = s + emis[torch.arange(T, device=emis.device), tags].sum()
        if T > 1: s = s + self.trans[tags[:-1], tags[1:]].sum()
        return s

    def _log_partition(self, emis):
        a = self.start + emis[0]
        for t in range(1, emis.size(0)):
            a = torch.logsumexp(a.unsqueeze(1) + self.trans, dim=0) + emis[t]
        return torch.logsumexp(a + self.end, dim=0)

    def nll(self, emis, tags):
        return (self._log_partition(emis) - self._gold_score(emis, tags)) / emis.size(0)

    def log_marginals(self, emis):
        """log P(y_t | x) for all t, via forward-backward. Differentiable.
            alpha_t(j) = log P(x_1..t, y_t=j)
            beta_t(i)  = log P(x_t+1..T | y_t=i)
            log P(y_t=j|x) = alpha_t(j) + beta_t(j) - logZ
        This is the sequence-aware soft target: it already integrates the transitions."""
        T = emis.size(0)
        a = [self.start + emis[0]]
        for t in range(1, T):
            a.append(torch.logsumexp(a[-1].unsqueeze(1) + self.trans, dim=0) + emis[t])
        b = [None]*T; b[T-1] = self.end
        for t in range(T-2, -1, -1):
            b[t] = torch.logsumexp(self.trans + (emis[t+1] + b[t+1]).unsqueeze(0), dim=1)
        logZ = torch.logsumexp(a[-1] + self.end, dim=0)
        return torch.stack([a[t] + b[t] for t in range(T)]) - logZ

    @torch.no_grad()
    def viterbi(self, emis):
        T, C = emis.shape
        score = self.start + emis[0]; ptrs = []
        for t in range(1, T):
            best, idx = (score.unsqueeze(1) + self.trans).max(dim=0)
            score = best + emis[t]; ptrs.append(idx)
        score = score + self.end
        cur = int(score.argmax()); path = [cur]
        for idx in reversed(ptrs): cur = int(idx[cur]); path.append(cur)
        return path[::-1]

class Head(nn.Module):
    def __init__(self, cfg, d_in):
        super().__init__()
        self.in_norm = nn.LayerNorm(d_in)
        self.bilstm = nn.LSTM(d_in, cfg.bilstm_hidden, cfg.bilstm_layers,
                              bidirectional=True, batch_first=True,
                              dropout=cfg.bilstm_dropout if cfg.bilstm_layers>1 else 0.0)
        self.drop = nn.Dropout(cfg.cls_dropout)
        self.emit = nn.Sequential(
            nn.Linear(2*cfg.bilstm_hidden, cfg.bilstm_hidden), nn.GELU(),
            nn.Dropout(cfg.cls_dropout), nn.Linear(cfg.bilstm_hidden, cfg.num_labels))
        self.crf = CRF(cfg.num_labels)
    def emissions(self, x):
        out, _ = self.bilstm(self.in_norm(x).unsqueeze(0))
        return self.emit(self.drop(out[0]))

# sanity check: marginals must exponentiate to 1 at every position
_h = Head(CFG, D_IN).to(DEVICE)
with torch.no_grad():
    _h.crf.trans.normal_(0, 0.5); _h.crf.start.normal_(0, 0.5); _h.crf.end.normal_(0, 0.5)
    _lm = _h.crf.log_marginals(torch.randn(17, NUM_LABELS, device=DEVICE))
    _s = _lm.exp().sum(-1)
print(f"forward-backward sanity: marginals sum to [{_s.min():.6f}, {_s.max():.6f}] "
      f"-> {'OK' if abs(_s.min()-1) < 1e-4 and abs(_s.max()-1) < 1e-4 else 'BUG'}")
del _h, _lm, _s
criterion = nn.CrossEntropyLoss(weight=class_weights)

forward-backward sanity: marginals sum to [0.999998, 1.000003] -> OK


In [ ]:
# CELL 6 — ONE TRAINER, used for teacher folds, baselines and students

def swa_crf(buf):
    c = CRF(NUM_LABELS).to(DEVICE)
    with torch.no_grad():
        c.trans.copy_(torch.tensor(np.mean([a[0] for a in buf], 0), dtype=torch.float32))
        c.start.copy_(torch.tensor(np.mean([a[1] for a in buf], 0), dtype=torch.float32))
        c.end.copy_(torch.tensor(np.mean([a[2] for a in buf], 0), dtype=torch.float32))
    return c

@torch.no_grad()
def emis_of(head, docs):
    head.eval()
    return {d["doc_id"]: head.emissions(d["x"].to(DEVICE)).cpu().numpy() for d in docs}

def decode_score(emis, docs, crf):
    yt, yp = [], []
    for d in docs:
        e = torch.tensor(emis[d["doc_id"]], dtype=torch.float32, device=DEVICE)
        yp.extend(crf.viterbi(e)); yt.extend(d["labels"])
    return macro_f1(yt, yp), yt, yp

def train_head(docs, epochs, seed, soft=None, alpha=1.0, temp=1.0,
               kd_mode="marginals", val_docs=None, patience=None):
    """soft: {doc_id: (T,C) teacher LOG-probabilities} or None.
       alpha: weight on the hard loss; (1-alpha) on the KD term."""
    seed_everything(seed)
    head = Head(CFG, D_IN).to(DEVICE)
    crf_ids = {id(p) for p in head.crf.parameters()}
    opt = torch.optim.AdamW([
        {"params": [p for p in head.parameters() if id(p) not in crf_ids],
         "lr": CFG.head_lr, "weight_decay": CFG.weight_decay},
        {"params": list(head.crf.parameters()), "lr": CFG.crf_lr, "weight_decay": 0.0}])
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)

    sdev, stest, scrf = (deque(maxlen=CFG.swa_window) for _ in range(3))
    best, bad = -1.0, 0
    order = list(range(len(docs)))
    for ep in range(epochs):
        head.train(); random.shuffle(order)
        for i in order:
            d = docs[i]
            x = d["x"].to(DEVICE)
            y = torch.tensor(d["labels"], dtype=torch.long, device=DEVICE)
            e = head.emissions(x)
            hard = head.crf.nll(e, y) + CFG.aux_ce_weight * criterion(e, y)
            loss = alpha * hard
            if soft is not None and d["doc_id"] in soft:
                tlog = torch.tensor(soft[d["doc_id"]], dtype=torch.float32, device=DEVICE)
                if kd_mode == "marginals":
                    slog = head.crf.log_marginals(e)
                    if temp != 1.0:
                        tlog = F.log_softmax(tlog / temp, dim=-1)
                        slog = F.log_softmax(slog / temp, dim=-1)
                    kd = (tlog.exp() * (tlog - slog)).sum(-1).mean()
                else:                                  # classic emission softmax KD
                    slog = F.log_softmax(e / temp, dim=-1)
                    tl   = F.log_softmax(tlog / temp, dim=-1)
                    kd = (tl.exp() * (tl - slog)).sum(-1).mean() * (temp ** 2)
                loss = loss + (1.0 - alpha) * kd
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(head.parameters(), CFG.max_grad_norm); opt.step()
        sch.step()

        sdev.append(emis_of(head, dev_docs)); stest.append(emis_of(head, test_docs))
        scrf.append(tuple(v.detach().cpu().numpy().copy()
                          for v in (head.crf.trans, head.crf.start, head.crf.end)))
        if val_docs is not None and patience is not None:
            f1, _, _ = decode_score(emis_of(head, val_docs), val_docs, head.crf)
            if f1 > best: best, bad = f1, 0
            else: bad += 1
            if bad >= patience: break

    c = swa_crf(scrf)
    dev_e  = {d["doc_id"]: np.mean([b[d["doc_id"]] for b in sdev], 0)  for d in dev_docs}
    test_e = {d["doc_id"]: np.mean([b[d["doc_id"]] for b in stest], 0) for d in test_docs}
    swa_f1, _, _ = decode_score(dev_e, dev_docs, c)
    return {"head": head, "crf": c, "dev_e": dev_e, "test_e": test_e,
            "best": best, "swa": swa_f1}

print("Trainer ready ✓")

Trainer ready ✓


In [ ]:
# CELL 7 — TEACHER: 5-fold CV inside the 80 training docs -> OUT-OF-FOLD soft targets

def folds_of(docs, k, seed=42):
    rng = random.Random(seed); f = [[] for _ in range(k)]
    for dom in ("CL", "IT"):
        idx = [i for i, d in enumerate(docs) if d["domain"] == dom]; rng.shuffle(idx)
        for j, i in enumerate(idx): f[j % k].append(i)
    return f

print("=" * 72); print("TEACHER — 5-fold, out-of-fold targets"); print("=" * 72)
fl = folds_of(train_docs, CFG.n_folds)
oof_soft, t_dev, t_test, t_crf, fold_f1 = {}, [], [], [], []
t_all = time.time()

for fi, held in enumerate(fl):
    t0 = time.time()
    hs = set(held)
    inner = [train_docs[i] for i in range(len(train_docs)) if i not in hs]
    held_docs = [train_docs[i] for i in held]
    # Fixed epoch count, no early stopping -> the held-out fold stays untouched
    res = train_head(inner, CFG.teacher_epochs, seed=1000 + fi)

    # the OOF soft targets: CRF posterior marginals on documents this model never saw
    with torch.no_grad():
        for d in held_docs:
            e = res["head"].emissions(d["x"].to(DEVICE))
            lm = (res["crf"].log_marginals(e) if CFG.kd_mode == "marginals" else e)
            oof_soft[d["doc_id"]] = lm.cpu().numpy()
    hf1, _, _ = decode_score(emis_of(res["head"], held_docs), held_docs, res["crf"])
    fold_f1.append(hf1)

    t_dev.append(res["dev_e"]); t_test.append(res["test_e"])
    t_crf.append(tuple(v.detach().cpu().numpy().copy()
                       for v in (res["crf"].trans, res["crf"].start, res["crf"].end)))
    del res["head"]; torch.cuda.empty_cache()
    print(f"  fold {fi+1}/{CFG.n_folds}: held-out F1={hf1:.4f}  {time.time()-t0:.0f}s")

assert len(oof_soft) == len(train_docs), (len(oof_soft), len(train_docs))
ens_crf   = swa_crf(t_crf)
ens_dev_e = {d["doc_id"]: np.mean([b[d["doc_id"]] for b in t_dev], 0)  for d in dev_docs}
ens_test_e= {d["doc_id"]: np.mean([b[d["doc_id"]] for b in t_test], 0) for d in test_docs}
ENS_F1, _, _ = decode_score(ens_dev_e, dev_docs, ens_crf)

print(f"\nTeacher built in {(time.time()-t_all)/60:.1f} min")
print(f"  per-fold held-out F1: mean {np.mean(fold_f1):.4f} +/- {np.std(fold_f1):.4f}")
print(f"  5-model ENSEMBLE on dev: {ENS_F1:.4f}")
print(f"  OOF soft targets for all {len(oof_soft)} training docs "
      f"({CFG.kd_mode}) — none from a model that saw its own document")
ex = oof_soft[train_docs[0]["doc_id"]]
if CFG.kd_mode == "marginals":
    p = np.exp(ex)
    print(f"  target entropy: {-(p*np.log(p+1e-12)).sum(-1).mean():.3f} nats "
          f"(a one-hot label is 0.000 — this is the extra signal)")

TEACHER — 5-fold, out-of-fold targets
  fold 1/5: held-out F1=0.4177  305s
  fold 2/5: held-out F1=0.4155  313s
  fold 3/5: held-out F1=0.3734  237s
  fold 4/5: held-out F1=0.4202  295s
  fold 5/5: held-out F1=0.4565  293s

Teacher built in 24.0 min
  per-fold held-out F1: mean 0.4167 +/- 0.0264
  5-model ENSEMBLE on dev: 0.4729
  OOF soft targets for all 80 training docs (marginals) — none from a model that saw its own document
  target entropy: 0.167 nats (a one-hot label is 0.000 — this is the extra signal)


In [ ]:
# CELL 8 — ARM 1: BASELINE (hard labels only) and ARM 2: STUDENT (hard + OOF soft)
#   Identical data, epochs, head, seeds and early-stopping rule. Only the loss differs.

print("=" * 72); print("ARM 1 — BASELINE, hard labels only"); print("=" * 72)
base_runs = []
for sd in CFG.seeds:
    t0 = time.time()
    r = train_head(train_docs, CFG.student_epochs, seed=sd,
                   soft=None, alpha=1.0,
                   val_docs=dev_docs, patience=CFG.patience)
    base_runs.append(r)
    print(f"  seed {sd}: best {r['best']:.4f} | swa {r['swa']:.4f} | {time.time()-t0:.0f}s")
    del r["head"]; torch.cuda.empty_cache()

print("\n" + "=" * 72)
print(f"ARM 2 — STUDENT, hard + OOF {CFG.kd_mode} (alpha={CFG.kd_alpha})")
print("=" * 72)
stud_runs = []
for sd in CFG.seeds:
    t0 = time.time()
    r = train_head(train_docs, CFG.student_epochs, seed=sd,
                   soft=oof_soft, alpha=CFG.kd_alpha, temp=CFG.kd_temp,
                   kd_mode=CFG.kd_mode, val_docs=dev_docs, patience=CFG.patience)
    stud_runs.append(r)
    print(f"  seed {sd}: best {r['best']:.4f} | swa {r['swa']:.4f} | {time.time()-t0:.0f}s")
    del r["head"]; torch.cuda.empty_cache()

B_swa = [r["swa"] for r in base_runs]; S_swa = [r["swa"] for r in stud_runs]
print(f"\nbaseline swa {np.mean(B_swa):.4f} +/- {np.std(B_swa):.4f}")
print(f"student  swa {np.mean(S_swa):.4f} +/- {np.std(S_swa):.4f}")

ARM 1 — BASELINE, hard labels only
  seed 42: best 0.4277 | swa 0.4132 | 221s
  seed 1337: best 0.4262 | swa 0.4320 | 266s
  seed 2024: best 0.4833 | swa 0.4552 | 314s

ARM 2 — STUDENT, hard + OOF marginals (alpha=0.5)
  seed 42: best 0.4511 | swa 0.4500 | 765s
  seed 1337: best 0.4511 | swa 0.4331 | 869s
  seed 2024: best 0.4750 | swa 0.4476 | 1054s

baseline swa 0.4335 +/- 0.0171
student  swa 0.4436 +/- 0.0075


In [ ]:
# CELL 9 — *** DID DISTILLATION HELP? *** the measurement cell

Bm, Bs = float(np.mean(B_swa)), float(np.std(B_swa))
Sm, Ss = float(np.mean(S_swa)), float(np.std(S_swa))

print("=" * 72)
print("CONTROLLED COMPARISON — same data, head, epochs, seeds, dev set")
print("=" * 72)
print(f"{'arm':<42}{'dev macro-F1':>18}")
print(f"{'1  baseline (hard labels only)':<42}{Bm:>10.4f} +/-{Bs:.3f}")
print(f"{'2  student (hard + OOF soft)':<42}{Sm:>10.4f} +/-{Ss:.3f}")
print(f"{'   teacher 5-model ensemble':<42}{ENS_F1:>10.4f}")

d_kd = Sm - Bm
pooled = float(np.sqrt((Bs**2 + Ss**2) / 2))
bar = max(CFG.DEV_NOISE, pooled) * 1.5
print(f"\nKD effect (student - baseline): {d_kd:+.4f}")
print(f"  measured dev noise floor (1 sd) : {CFG.DEV_NOISE:.3f}")
print(f"  pooled seed spread (1 sd)       : {pooled:.3f}")
print(f"  bar for a real effect (1.5 sd)  : {bar:.3f}")
verdict = ("DISTILLATION HELPED" if d_kd > bar else
           "DISTILLATION HURT"   if d_kd < -bar else
           "NO DETECTABLE DIFFERENCE")
print(f"\n  >>> {verdict}")

# How much of the ensemble's advantage did the student recover?
gap = ENS_F1 - Bm
if abs(gap) > 1e-6:
    print(f"\nensemble advantage over baseline : {gap:+.4f}")
    print(f"recovered by the student         : {100*d_kd/gap:+.1f}% of it")
    print("(this is the number to quote in the report — compression retention)")
print(f"\nbest submission available: "
      f"{'ENSEMBLE' if ENS_F1 >= max(Sm, Bm) else ('STUDENT' if Sm >= Bm else 'BASELINE')}")
print("Submit the ensemble unless the student actually beat it — distillation is a")
print("compression technique and you do not need compression, you need accuracy.")

# Does KD fix the overconfidence we measured (macro P 0.51 vs R 0.40)?
from sklearn.metrics import precision_recall_fscore_support as prf
for tag, e, c in (("baseline", base_runs[0]["dev_e"], base_runs[0]["crf"]),
                  ("student ", stud_runs[0]["dev_e"], stud_runs[0]["crf"]),
                  ("ensemble", ens_dev_e, ens_crf)):
    _, yt, yp = decode_score(e, dev_docs, c)
    P, R, _, _ = prf(yt, yp, labels=list(range(NUM_LABELS)),
                     average="macro", zero_division=0)
    print(f"  {tag}: macro P={P:.4f}  R={R:.4f}  gap={P-R:+.4f}")
print("A smaller P-R gap means the soft targets did reduce overconfidence, which is")
print("the mechanism this experiment is actually testing.")

CONTROLLED COMPARISON — same data, head, epochs, seeds, dev set
arm                                             dev macro-F1
1  baseline (hard labels only)                0.4335 +/-0.017
2  student (hard + OOF soft)                  0.4436 +/-0.007
   teacher 5-model ensemble                   0.4729

KD effect (student - baseline): +0.0101
  measured dev noise floor (1 sd) : 0.026
  pooled seed spread (1 sd)       : 0.013
  bar for a real effect (1.5 sd)  : 0.039

  >>> NO DETECTABLE DIFFERENCE

ensemble advantage over baseline : +0.0394
recovered by the student         : +25.7% of it
(this is the number to quote in the report — compression retention)

best submission available: ENSEMBLE
Submit the ensemble unless the student actually beat it — distillation is a
compression technique and you do not need compression, you need accuracy.
  baseline: macro P=0.4996  R=0.3919  gap=+0.1077
  student : macro P=0.5248  R=0.4341  gap=+0.0907
  ensemble: macro P=0.5286  R=0.4549  gap=+0.0737
A 

In [ ]:
# CELL 10 — ABLATION: alpha sweep and marginals vs emissions (1 seed each, ~5 min)

print("alpha = weight on the HARD loss. alpha=1.0 is the baseline, 0.0 is soft-only.")
abl = []
for mode in ("marginals", "emissions"):
    # the teacher targets were saved in CFG.kd_mode form; only that mode is directly usable
    if mode != CFG.kd_mode:
        print(f"\n[{mode}] skipped — rerun CELL 7 with kd_mode='{mode}' to test it")
        continue
    print(f"\n[{mode}]")
    for a in (1.0, 0.7, 0.5, 0.3, 0.0):
        r = train_head(train_docs, CFG.student_epochs, seed=CFG.seeds[0],
                       soft=oof_soft, alpha=a, temp=CFG.kd_temp, kd_mode=mode,
                       val_docs=dev_docs, patience=CFG.patience)
        abl.append({"mode": mode, "alpha": a, "swa": r["swa"], "best": r["best"]})
        print(f"  alpha={a:.1f}  swa={r['swa']:.4f}  best={r['best']:.4f}")
        del r["head"]; torch.cuda.empty_cache()

if abl:
    bestab = max(abl, key=lambda z: z["swa"])
    print(f"\nbest: {bestab['mode']} alpha={bestab['alpha']} -> {bestab['swa']:.4f}")
    print(f"vs baseline {Bm:.4f}  (delta {bestab['swa']-Bm:+.4f}, bar {bar:.3f})")
    print("\nNOTE: this sweep picks the best of 5 alphas on a 10-doc dev set, so the")
    print("winner is biased upward by selection. Treat it as a direction, not a score.")

alpha = weight on the HARD loss. alpha=1.0 is the baseline, 0.0 is soft-only.

[marginals]
  alpha=1.0  swa=0.4132  best=0.4277


KeyboardInterrupt: 

In [ ]:
# CELL 11 — PER-CLASS REPORT + SUBMISSION from whichever arm won

cands = [("ensemble", ENS_F1, ens_dev_e, ens_test_e, ens_crf),
         ("student",  Sm, stud_runs[int(np.argmax(S_swa))]["dev_e"],
                          stud_runs[int(np.argmax(S_swa))]["test_e"],
                          stud_runs[int(np.argmax(S_swa))]["crf"]),
         ("baseline", Bm, base_runs[int(np.argmax(B_swa))]["dev_e"],
                          base_runs[int(np.argmax(B_swa))]["test_e"],
                          base_runs[int(np.argmax(B_swa))]["crf"])]
win, wf1, dev_e, test_e, crf_use = max(cands, key=lambda z: z[1])
print(f"Submitting arm: {win} (dev {wf1:.4f})\n")

_, yt, yp = decode_score(dev_e, dev_docs, crf_use)
print(classification_report(yt, yp, labels=list(range(NUM_LABELS)),
                            target_names=[ID2LABEL[i] for i in range(NUM_LABELS)],
                            digits=4, zero_division=0))
sup = np.bincount(yt, minlength=NUM_LABELS)
ok = [i for i in range(NUM_LABELS) if sup[i] > 0]
print(f"supported-only ({len(ok)}/{NUM_LABELS}): "
      f"{f1_score(yt, yp, labels=ok, average='macro', zero_division=0):.4f}")

predictions = {}
for d in test_docs:
    e = torch.tensor(test_e[d["doc_id"]], dtype=torch.float32, device=DEVICE)
    path = crf_use.viterbi(e)
    assert len(path) == len(d["sentences"]), d["doc_id"]
    predictions[d["doc_id"]] = [ID2LABEL[int(p)] for p in path]
assert len(predictions) == len(test_docs)
assert all(l in LABEL2ID for v in predictions.values() for l in v)

pp = OUTPUT_DIR / "prediction.json"
json.dump(predictions, open(pp, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
zp = OUTPUT_DIR / "submission.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf: zf.write(pp, "prediction.json")
print(f"\nSaved {zp}  (arm {win})")

pc = Counter(l for v in predictions.values() for l in v)
ptot, ttot = sum(pc.values()), sum(cnt.values())
print(f"Predicted {len(pc)}/{NUM_LABELS} labels")
for i in sorted(range(NUM_LABELS), key=lambda i: -cnt.get(i, 0)):
    l = ID2LABEL[i]
    print(f"  {l:<26}{pc.get(l,0):>6}{100*pc.get(l,0)/ptot:>7.2f}%  train {100*cnt.get(i,0)/ttot:.2f}%")

json.dump({"baseline_mean": Bm, "baseline_std": Bs,
           "student_mean": Sm, "student_std": Ss,
           "ensemble": float(ENS_F1), "kd_delta": float(d_kd),
           "bar": float(bar), "verdict": verdict,
           "fold_f1": [float(f) for f in fold_f1],
           "kd_mode": CFG.kd_mode, "kd_alpha": CFG.kd_alpha, "kd_temp": CFG.kd_temp,
           "ablation": abl, "submitted_arm": win},
          open(OUTPUT_DIR / "kd_summary.json", "w"), indent=2)
print(f"\nSummary -> {OUTPUT_DIR/'kd_summary.json'}")

Submitting arm: ensemble (dev 0.4729)

                        precision    recall  f1-score   support

                  Fact     0.7672    0.7275    0.7468       444
                 Issue     0.7500    0.4000    0.5217        15
    ArgumentPetitioner     0.6620    0.4700    0.5497       400
    ArgumentRespondent     0.5301    0.5184    0.5242       272
               Statute     0.9135    0.7917    0.8482       120
               Dissent     0.0000    0.0000    0.0000         0
   PrecedentReliedUpon     0.7337    0.5315    0.6164       254
PrecedentNotReliedUpon     0.2329    0.3400    0.2764        50
    PrecedentOverruled     0.0000    0.0000    0.0000         0
    RulingByLowerCourt     0.3741    0.5392    0.4418       102
    RatioOfTheDecision     0.5780    0.8734    0.6957       403
  RulingByPresentCourt     0.6875    0.4490    0.5432        49
                  None     0.6429    0.2727    0.3830        33

              accuracy                         0.6270      2142